# Reusable Template – Binary Logistic Regression (2-feature or mapped)
Copy this notebook for any new binary classification problem that can be visualised in 2-D (or after a feature map).

Replace the data-loading cell, then run the rest.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy
from sklearn.linear_model import LogisticRegression
%matplotlib inline

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def compute_cost_vec(X, y, w, b, lambda_=0):
    m = X.shape[0]
    f = np.clip(sigmoid(X @ w + b), 1e-15, 1-1e-15)
    cost = -np.mean(y*np.log(f) + (1-y)*np.log(1-f))
    if lambda_ > 0:
        cost += (lambda_/(2*m)) * np.sum(w**2)
    return cost

def compute_gradient_vec(X, y, w, b, lambda_=0):
    m = X.shape[0]
    err = sigmoid(X @ w + b) - y
    dj_dw = (X.T @ err)/m + (lambda_/m)*w if lambda_ else (X.T @ err)/m
    return np.mean(err), dj_dw

def gradient_descent(X, y, w, b, alpha, iters, lambda_=0, print_every=None):
    w = w.astype(float).copy(); b = float(b)
    if print_every is None: print_every = max(1, iters//10)
    for i in range(iters):
        db, dw = compute_gradient_vec(X, y, w, b, lambda_)
        w -= alpha*dw; b -= alpha*db
        if i % print_every == 0:
            print(f"Iter {i:6d}  cost={compute_cost_vec(X,y,w,b,lambda_):.4f}")
    return w, b

def predict(X, w, b, thr=0.5):
    return (sigmoid(X @ w + b) >= thr).astype(int)


## 1. Load your data
Expect `X` shape (m, n) and binary `y` shape (m,).


In [ ]:
# EXAMPLE – replace with your own loader
# from utils import load_data
# X, y = load_data("data/ex2data1.txt")
# For mapped features: X = map_feature(X[:,0], X[:,1])

# Placeholder random data for template testing
np.random.seed(0)
X = np.random.randn(100, 2)
y = (X[:,0] + X[:,1] > 0).astype(float)
print(X.shape, y.shape)


## 2. Train from-scratch model


In [ ]:
w, b = gradient_descent(X, y, np.zeros(X.shape[1]), 0.0,
                        alpha=0.1, iters=5000, lambda_=0.1)
print("w, b =", w, b)
print("Accuracy =", np.mean(predict(X,w,b)==y)*100, "%")


## 3. sklearn baseline


In [ ]:
clf = LogisticRegression(penalty="l2", C=10, max_iter=1000)
clf.fit(X, y)
print("sklearn accuracy:", clf.score(X,y)*100)


## 4. Quick simulation – accuracy vs λ


In [ ]:
lambdas = np.logspace(-3, 2, 8)
accs = []
for lam in lambdas:
    ww, bb = gradient_descent(X, y, np.zeros(X.shape[1]), 0., 0.1, 3000, lambda_=lam, print_every=10000)
    accs.append(np.mean(predict(X,ww,bb)==y)*100)
plt.semilogx(lambdas, accs, "o-")
plt.xlabel("λ"); plt.ylabel("Accuracy %"); plt.title("λ sensitivity"); plt.grid(True); plt.show()
